<a href="https://colab.research.google.com/github/hitesh771/Building-Segmentation-Detectron2/blob/main/1_Training_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install Detectron2
# This command will build Detectron2 from source to match your Colab environment
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

# 2. Import all the libraries (your original code)
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

import numpy as np
import cv2
import random
from google.colab.patches import cv2_imshow

from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog

print("✅ Detectron2 installation and imports successful!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Let's check if it can see your new folder
# This path should now work
!ls "/content/drive/MyDrive/WHOdataset/WHU"

In [ ]:
# --- 1. Install the progress bar library ---
# This is the only extra library we actually need
!pip install -q tqdm

# --- 2. THE CONVERSION SCRIPT (with updated paths) ---
import os
import cv2
import numpy as np
import json
from tqdm import tqdm # A nice progress bar

# --- DEFINE YOUR FOLDER PATHS (UPDATED from your screenshot) ---
# This path points to your new folder in Google Drive
BASE_DIR = "/content/drive/MyDrive/WHOdataset/WHU/train"
# Use the *exact* case-sensitive names from your screenshot
IMAGE_DIR = os.path.join(BASE_DIR, "Image")
MASK_DIR = os.path.join(BASE_DIR, "Mask")

# --- DEFINE THE OUTPUT JSON FILE ---
# This is the "answer" file we are creating
COCO_JSON_PATH = os.path.join(BASE_DIR, "_annotations.json")

# --- Initialize the COCO JSON structure ---
coco_output = {
    "info": {},
    "licenses": [],
    "categories": [{"id": 1, "name": "building", "supercategory": "building"}],
    "images": [],
    "annotations": []
}

image_id_counter = 1
annotation_id_counter = 1

print(f"Starting conversion... this may take a minute.")

# --- Loop through every image in your IMAGE_DIR ---
for image_filename in tqdm(sorted(os.listdir(IMAGE_DIR))):
    if not image_filename.endswith(('.png', '.jpg', '.tif')):
        continue

    # --- 1. Add Image Info ---
    image_path = os.path.join(IMAGE_DIR, image_filename)
    img = cv2.imread(image_path)

    # Check if image loaded correctly
    if img is None:
        print(f"\nWarning: Could not read image {image_path}. Skipping.")
        continue

    height, width, _ = img.shape

    image_info = {
        "id": image_id_counter,
        "file_name": image_filename, # Just the filename
        "width": width,
        "height": height
    }
    coco_output["images"].append(image_info)

    # --- 2. Find its matching mask ---
    mask_path = os.path.join(MASK_DIR, image_filename)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        print(f"\nWarning: No matching mask found for {image_filename}. Skipping.")
        continue

    # --- 3. Find outlines (contours) for each building ---
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # --- 4. Convert each contour to COCO format ---
    for contour in contours:
        segmentation = contour.flatten().tolist()

        if len(segmentation) < 6: # A valid polygon needs at least 3 points (6 values)
            continue

        bbox = cv2.boundingRect(contour)
        area = cv2.contourArea(contour)

        annotation_info = {
            "id": annotation_id_counter,
            "image_id": image_id_counter,
            "category_id": 1, # "building"
            "segmentation": [segmentation],
            "bbox": list(bbox),
            "area": float(area),
            "iscrowd": 0
        }
        coco_output["annotations"].append(annotation_info)
        annotation_id_counter += 1

    image_id_counter += 1

# --- 5. Save the final JSON file ---
with open(COCO_JSON_PATH, 'w') as f:
    json.dump(coco_output, f)
print(f"\n✅ SUCCESS! Created '{COCO_JSON_PATH}'")

In [ ]:
# --- 1. Re-install Detectron2 ---
print("Installing Detectron2...")
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

# --- 2. Mount your Google Drive ---
print("\nConnecting to Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

print("✅ Environment is ready.")

In [ ]:
# --- 1. Import all libraries ---
print("Importing libraries...")
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer
import os
import random
import cv2
from google.colab.patches import cv2_imshow
import detectron2 # Import the main library

# --- 2. SET THE FOLDER PATHS (from your screenshot) ---
dataset_name = "whu_buildings_train"
dataset_json_path = "/content/drive/MyDrive/WHOdataset/WHU/train/_annotations.json"
dataset_image_dir = "/content/drive/MyDrive/WHOdataset/WHU/train/Image"

# --- 3. Safety check ---
if dataset_name in DatasetCatalog.list():
    DatasetCatalog.remove(dataset_name)

# --- 4. Register the dataset (FIXED - no stray underscore) ---
try:
    register_coco_instances(dataset_name,
                            {},
                            dataset_json_path,
                            dataset_image_dir)

    print(f"✅ SUCCESS! Dataset '{dataset_name}' is registered.")

except FileNotFoundError:
    print("--- ❌ ERROR ---")
    print(f"Could not find the file at: {dataset_json_path}")


# --- 5. (SANITY CHECK) ---
print("\nRunning sanity check... displaying 1 random training image:")
try:
    building_metadata = MetadataCatalog.get(dataset_name)
    dataset_dicts = DatasetCatalog.get(dataset_name)

    d = random.choice(dataset_dicts)
    img = cv2.imread(d["file_name"])
    v = Visualizer(img[:, :, ::-1], metadata=building_metadata, scale=0.8)
    out = v.draw_dataset_dict(d)
    print(f"Displaying {d['file_name']}")
    cv2_imshow(out.get_image()[:, :, ::-1])
except Exception as e:
    print(f"\n--- ❌ ERROR DURING SANITY CHECK ---")
    print(f"Could not visualize the data. Error: {e}")

In [ ]:
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
import os

# --- 1. Create a new config ---
cfg = get_cfg()

# --- 2. Load the base model architecture ---
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))

# --- 3. Set the model to use pre-trained COCO weights ---
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

# --- 4. Point to our registered dataset ---
cfg.DATASETS.TRAIN = ("whu_buildings_train",)
cfg.DATASETS.TEST = () # No validation set for now

# --- 5. Set training parameters ---
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 1000
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128

# --- 6. Set the number of classes ---
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # (Just "building")

# --- 7. Set the output directory ---
cfg.OUTPUT_DIR = "./output_whu_buildings"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("✅ Training configuration is set. Ready to train!")

# --- 8. RUN THE TRAINING ---
print("Starting training... This will take a while.")

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

print("🎉🎉🎉 TRAINING COMPLETE! 🎉🎉🎉")

In [ ]:
# This copies your new model from the temporary Colab folder
# to your permanent Google Drive.

!cp "/content/output_whu_buildings/model_final.pth" "/content/drive/MyDrive/my_building_model.pth"

print("✅ SUCCESS! Your trained model is now permanently saved in your Google Drive.")
print("The file is at: /content/drive/MyDrive/my_building_model.pth")

# This copies your model and your metrics to your Google Drive
!cp "/content/output_whu_buildings/model_final.pth" "/content/drive/MyDrive/my_building_model.pth"
!cp "/content/output_whu_buildings/metrics.json" "/content/drive/MyDrive/training_metrics.json"

print("✅ SUCCESS! Your model and metrics are now permanently saved in your Google Drive.")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the metrics file from your Google Drive
metrics_path = "/content/drive/MyDrive/training_metrics.json"

# The metrics file is in a "JSON-lines" format
data = []
with open(metrics_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))

# Convert to a pandas DataFrame for easy plotting
df = pd.DataFrame(data)

# --- Create the Loss Curve Plot ---
plt.figure(figsize=(10, 6))
# We skip the first 20 iterations as they are usually
# very high and skew the plot.
plt.plot(df['iteration'][20:], df['total_loss'][20:])
plt.title('Training Loss Curve (Total Loss vs. Iterations)')
plt.xlabel('Iteration')
plt.ylabel('Total Loss')
plt.grid(True)

# --- Save the plot for your PPT ---
plot_save_path = "/content/drive/MyDrive/training_loss_curve.png"
plt.savefig(plot_save_path)

print(f"✅ Loss curve plot saved to your Google Drive at: {plot_save_path}")
print("You can download this .png file and your 'training_metrics.json' file from your Google Drive for your presentation.")
plt.show()